# 1.4 低秩分解（Low-Rank Factorization）

## 什么是低秩分解？

低秩分解将大型权重矩阵分解为两个或多个低秩矩阵的乘积，减少参数量和计算量。核心思想是：大多数权重矩阵的有效秩远小于其名义维度，存在大量冗余。

## 为什么用低秩分解？

1. **参数冗余**：神经网络权重矩阵通常是低秩的或近似低秩的，SVD分解后大部分奇异值接近0，截断它们对精度影响极小。
2. **计算量降低**：原始矩阵乘法$O(mn)$分解为两次小矩阵乘法$O(r(m+n))$，当$r \ll \min(m,n)$时计算量大幅减少。
3. **与LoRA的关联**：LoRA本质上就是一种低秩分解——将权重增量$\Delta W$分解为$AB$，仅训练低秩部分。
4. **端侧友好**：分解后的小矩阵更适合端侧设备的有限缓存和计算能力。

## 低秩分解的数学原理

对于权重矩阵$W \in \mathbb{R}^{m \times n}$，假设其有效秩为$r$（$r \ll \min(m,n)$），则：

$$W \approx AB, \quad A \in \mathbb{R}^{m \times r}, B \in \mathbb{R}^{r \times n}$$

参数量从$mn$降至$r(m+n)$，压缩比为$\frac{mn}{r(m+n)}$。当$r \ll \min(m,n)$时，压缩比接近$\frac{\min(m,n)}{r}$。

其中：
- $W$：原始权重矩阵
- $A$：左因子矩阵
- $B$：右因子矩阵
- $r$：分解的秩，控制压缩比与精度的权衡

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

---
### SVD分解（Singular Value Decomposition）

#### 什么是SVD分解？

对权重矩阵进行奇异值分解 $W = U\Sigma V^T$，保留前r个最大奇异值，得到 $W \approx U_r \Sigma_r V_r^T$，参数量从 $mn$ 降至 $r(m+n)$。

这是最经典的低秩分解方法，通过截断小奇异值来压缩矩阵。SVD提供了在Frobenius范数下的最优低秩近似（Eckart-Young定理）。

#### 为什么SVD分解有效？

1. **最优低秩近似**：Eckart-Young定理保证截断SVD给出Frobenius范数下的最优秩r近似。
2. **能量集中**：奇异值按大小排列，前几个奇异值通常包含大部分"能量"（信息），截断小奇异值损失很小。
3. **无需训练**：SVD是纯数学分解，不需要训练数据或反向传播。

#### SVD分解的数学公式

**完整SVD**：
$$W = U \Sigma V^T = \sum_{i=1}^{\min(m,n)} \sigma_i u_i v_i^T$$

**截断SVD（保留前r个奇异值）**：
$$W \approx W_r = U_r \Sigma_r V_r^T = \sum_{i=1}^{r} \sigma_i u_i v_i^T$$

**分解为两个矩阵**：
$$A = U_r \Sigma_r \in \mathbb{R}^{m \times r}, \quad B = V_r^T \in \mathbb{R}^{r \times n}$$

**能量保留比**：
$$\text{Energy}(r) = \frac{\sum_{i=1}^{r} \sigma_i^2}{\sum_{i=1}^{\min(m,n)} \sigma_i^2}$$

其中：
- $W \in \mathbb{R}^{m \times n}$：原始权重矩阵
- $U \in \mathbb{R}^{m \times m}$：左奇异向量矩阵
- $\Sigma \in \mathbb{R}^{m \times n}$：奇异值对角矩阵，$\sigma_1 \geq \sigma_2 \geq ... \geq 0$
- $V \in \mathbb{R}^{n \times n}$：右奇异向量矩阵
- $r$：保留的奇异值数量（秩）
- $\sigma_i$：第$i$个奇异值，越大表示该方向越重要
- 能量保留比通常取90%-99%来确定合适的秩$r$

In [ ]:
def svd_compress(weight: torch.Tensor, rank: int) -> tuple:
    """SVD低秩压缩"""
    U, S, Vh = torch.linalg.svd(weight, full_matrices=False)
    U_r = U[:, :rank]
    S_r = S[:rank]
    Vh_r = Vh[:rank, :]
    A = U_r @ torch.diag(S_r)
    B = Vh_r
    return A, B, S

def svd_decompress(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """SVD低秩重建"""
    return A @ B

m, n = 256, 512
rank_true = 32
U_true = torch.randn(m, rank_true)
V_true = torch.randn(rank_true, n)
W_lowrank = U_true @ V_true + torch.randn(m, n) * 0.01

W_full = torch.randn(m, n)

print(f"=== SVD低秩压缩 ===")
print(f"原始矩阵: {m}x{n} = {m*n} 参数")

for W, name in [(W_lowrank, "低秩矩阵(有效秩≈32)"), (W_full, "稠密随机矩阵")]:
    print(f"\n--- {name} ---")
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    total_energy = (S ** 2).sum()
    cumulative = torch.cumsum(S ** 2, dim=0) / total_energy
    for threshold in [0.9, 0.95, 0.99]:
        r = (cumulative < threshold).sum().item() + 1
        r = min(r, min(m, n))
        A, B, _ = svd_compress(W, rank=r)
        W_recon = svd_decompress(A, B)
        mse = F.mse_loss(W, W_recon)
        param_ratio = (m * r + r * n) / (m * n)
        print(f"  {threshold:.0%}能量保留: rank={r}, 参数比={param_ratio:.2%}, MSE={mse:.6f}")

ranks = [8, 16, 32, 64, 128]
print(f"\n--- 低秩矩阵不同rank的压缩效果 ---")
for r in ranks:
    A, B, _ = svd_compress(W_lowrank, rank=r)
    W_recon = svd_decompress(A, B)
    mse = F.mse_loss(W_lowrank, W_recon)
    cos = F.cosine_similarity(W_lowrank.unsqueeze(0), W_recon.unsqueeze(0)).mean()
    param_ratio = (m * r + r * n) / (m * n)
    print(f"  rank={r:<4} 参数比={param_ratio:.2%} MSE={mse:.6f} 余弦相似度={cos:.6f}")

### SVD压缩在Transformer线性层中的应用

#### 什么是SVD压缩在Transformer中的应用？

将SVD分解应用于Transformer中的线性层（QKV投影、输出投影、FFN层），用两个小矩阵替代原始大矩阵，实现模型压缩。

#### 为什么对Transformer线性层做SVD压缩？

1. **参数集中**：Transformer的大部分参数在线性层（约70%在FFN层），压缩线性层收益最大
2. **低秩特性**：FFN层的中间维度通常远大于隐藏维度，存在大量冗余
3. **精度可控**：通过选择保留的奇异值数量，精确控制压缩比与精度权衡

#### 数学原理

对于线性层$y = Wx + b$，$W \in \mathbb{R}^{m \times n}$：
$$W \approx U_r \Sigma_r V_r^T = A \cdot B$$
其中$A = U_r \Sigma_r \in \mathbb{R}^{m \times r}$，$B = V_r^T \in \mathbb{R}^{r \times n}$

推理时：$y = ABx + b$，两次小矩阵乘法替代一次大矩阵乘法
- 原始计算量：$O(mn)$
- 压缩后计算量：$O(r(m+n))$
- 压缩比：$\frac{mn}{r(m+n)}$

In [ ]:
import copy


class SVDCompressedLinear(nn.Module):
    """SVD压缩的线性层"""
    def __init__(self, A: torch.Tensor, B: torch.Tensor, bias: torch.Tensor = None):
        super().__init__()
        self.A = nn.Parameter(A)
        self.B = nn.Parameter(B)
        self.bias = nn.Parameter(bias) if bias is not None else None

    def forward(self, x):
        out = x @ self.B.T @ self.A.T
        if self.bias is not None:
            out = out + self.bias
        return out

    @staticmethod
    def from_linear(linear: nn.Linear, rank: int):
        """从标准线性层创建SVD压缩层"""
        W = linear.weight.data
        A, B, _ = svd_compress(W, rank=rank)
        return SVDCompressedLinear(A, B, linear.bias.data if linear.bias is not None else None)

class SmallTransformer(nn.Module):
    def __init__(self, dim=64, n_heads=4, n_classes=10):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.ln2(x))
        return self.head(x.mean(dim=1))

dim = 64
model = SmallTransformer(dim=dim, n_heads=4, n_classes=10)
original_params = sum(p.numel() for p in model.parameters())

x_test = torch.randn(32, 8, dim)
y_test = torch.randint(0, 10, (32,))

with torch.no_grad():
    out_orig = model(x_test)
    acc_orig = (out_orig.argmax(1) == y_test).float().mean()

for rank_ratio in [0.25, 0.5, 0.75]:
    model_copy = copy.deepcopy(model)
    compressed_params = 0
    for name, module in model_copy.named_modules():
        if isinstance(module, nn.Linear) and module.weight.dim() == 2:
            out_f, in_f = module.weight.shape
            rank = max(1, int(min(out_f, in_f) * rank_ratio))
            compressed_layer = SVDCompressedLinear.from_linear(module, rank=rank)
            parent_name = '.'.join(name.split('.')[:-1])
            child_name = name.split('.')[-1]
            parent = model_copy
            for part in parent_name.split('.'):
                if part:
                    parent = getattr(parent, part)
            setattr(parent, child_name, compressed_layer)

    compressed_params = sum(p.numel() for p in model_copy.parameters())
    with torch.no_grad():
        out_comp = model_copy(x_test)
        acc_comp = (out_comp.argmax(1) == y_test).float().mean()
        output_diff = (out_orig - out_comp).norm() / out_orig.norm() * 100

    print(f"rank比例={rank_ratio:.0%}: 参数量={compressed_params:,}({compressed_params/original_params:.1%}), "
          f"输出差异={output_diff:.2f}%, 准确率={acc_comp:.4f}")

print(f"\n原始参数量: {original_params:,}")

---
### Tucker分解

#### 什么是Tucker分解？

Tucker分解是对高维张量进行多模式分解的方法，保留各模式的主成分，适合多维权重张量（如卷积核4D张量）的压缩。对于2D矩阵，Tucker-2分解等价于截断SVD。

#### 为什么用Tucker分解？

1. **高维张量压缩**：卷积核是4D张量（输出通道×输入通道×高×宽），SVD只能处理2D矩阵，Tucker可以处理任意维度的张量。
2. **各维度独立压缩**：Tucker分解允许每个维度选择不同的秩，比SVD更灵活。
3. **保留空间结构**：对卷积核的空间维度单独分解，可以保留空间局部性。

#### Tucker分解的数学公式

Tucker分解将张量 $\mathcal{X} \in \mathbb{R}^{I_1 \times I_2 \times \cdots \times I_N}$ 分解为核心张量 $\mathcal{G}$ 和各模式的因子矩阵：
$$\mathcal{X} \approx \mathcal{G} \times_1 U^{(1)} \times_2 U^{(2)} \cdots \times_N U^{(N)}$$

其中：
- $\mathcal{X}$：原始N维张量
- $\mathcal{G} \in \mathbb{R}^{R_1 \times R_2 \times \cdots \times R_N}$：核心张量，$R_i \leq I_i$
- $U^{(i)} \in \mathbb{R}^{I_i \times R_i}$：第$i$个模式的因子矩阵
- $\times_i$：第$i$个模式的张量-矩阵乘法
- $R_i$：第$i$个模式的Tucker秩，控制该维度的压缩程度
- 参数量从$\prod I_i$降至$\prod R_i + \sum I_i R_i$

In [ ]:
def tucker_decompose_2d(weight: torch.Tensor, rank_ratio: float = 0.5) -> tuple:
    """对2D权重矩阵进行Tucker-2分解（即截断SVD的推广）"""
    m, n = weight.shape
    rank_m = max(1, int(m * rank_ratio))
    rank_n = max(1, int(n * rank_ratio))

    U, S, Vh = torch.linalg.svd(weight, full_matrices=False)
    U_r = U[:, :rank_m]
    S_r = S[:min(rank_m, rank_n)]
    Vh_r = Vh[:rank_n, :]

    core = torch.diag(S_r[:min(rank_m, rank_n)])
    if rank_m > rank_n:
        core = torch.cat([core, torch.zeros(rank_m - rank_n, rank_n)], dim=0)
    elif rank_n > rank_m:
        core = torch.cat([core, torch.zeros(rank_m, rank_n - rank_m)], dim=1)

    return U_r, core, Vh_r

def tucker_reconstruct(U, core, Vh):
    """Tucker重建"""
    return U @ core @ Vh

weight = torch.randn(256, 512)

print(f"=== Tucker分解 ===")
for ratio in [0.25, 0.5, 0.75]:
    U, core, Vh = tucker_decompose_2d(weight, rank_ratio=ratio)
    W_recon = tucker_reconstruct(U, core, Vh)
    mse = F.mse_loss(weight, W_recon)
    orig_params = weight.numel()
    comp_params = U.numel() + core.numel() + Vh.numel()
    print(f"rank比例={ratio:.0%}: 核心形状={list(core.shape)}, "
          f"参数比={comp_params/orig_params:.2%}, MSE={mse:.6f}")

print(f"\nTucker分解 vs SVD: Tucker允许各维度独立选择秩，更灵活")
print(f"对于2D矩阵，Tucker-2等价于截断SVD")
print(f"对于高维张量(如Conv权重4D)，Tucker分解优势更明显")

---
### 低秩重参数化（LoRA-style Factorization）

#### 什么是LoRA式低秩分解？

冻结原始权重W，添加低秩增量 ΔW = AB，其中A∈R^{m×r}, B∈R^{r×n}。推理时可将AB合并回W，无额外推理开销。虽主要用于微调，但其低秩思想可用于压缩。

#### 为什么LoRA式分解有效？

1. **零推理开销**：合并后$W' = W + \frac{\alpha}{r}AB$，与原始线性层完全等价，无额外计算。
2. **训练高效**：仅训练$r(m+n)$个参数（通常<1%总参数），显存需求大幅降低。
3. **多任务切换**：同一基座模型+不同LoRA适配器，可按需切换任务，无需加载多个完整模型。

#### LoRA的数学公式

$$h = xW^T + x \cdot \frac{\alpha}{r} \cdot A^T B^T = x(W + \frac{\alpha}{r}AB)^T$$

**合并后**：
$$W' = W + \frac{\alpha}{r}AB$$

其中：
- $W \in \mathbb{R}^{m \times n}$：原始冻结权重
- $A \in \mathbb{R}^{r \times n}$：LoRA下投影矩阵，可训练，随机初始化
- $B \in \mathbb{R}^{m \times r}$：LoRA上投影矩阵，可训练，零初始化
- $r$：LoRA秩，远小于$m$和$n$
- $\alpha$：缩放因子，控制LoRA增量的影响强度
- $\frac{\alpha}{r}$：缩放系数，确保不同rank下适配器的输出量级一致
- $B$零初始化确保训练开始时$\Delta W = 0$，不改变原始模型输出

In [ ]:
class LoRALinear(nn.Module):
    """LoRA线性层：原始权重冻结 + 低秩适配"""
    def __init__(self, in_features, out_features, rank=8, alpha=16, merge=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.merge = merge

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01, requires_grad=False)
        self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        self.merged = False

    def merge_weights(self):
        """将LoRA权重合并到主权重，推理时无额外开销"""
        if not self.merged:
            self.weight.data += self.scaling * (self.lora_B @ self.lora_A)
            self.merged = True

    def forward(self, x):
        if self.merged:
            return F.linear(x, self.weight, self.bias)
        result = F.linear(x, self.weight, self.bias)
        result += (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return result

in_f, out_f = 512, 256
lora_layer = LoRALinear(in_f, out_f, rank=8, alpha=16)

x = torch.randn(4, in_f)

with torch.no_grad():
    out_unmerged = lora_layer(x)

lora_layer.merge_weights()
with torch.no_grad():
    out_merged = lora_layer(x)

merge_diff = (out_unmerged - out_merged).norm() / out_unmerged.norm() * 100

base_params = in_f * out_f
lora_params = in_f * 8 + out_f * 8

print(f"=== LoRA低秩重参数化 ===")
print(f"原始权重参数: {base_params:,}")
print(f"LoRA参数(rank=8): {lora_params:,} ({lora_params/base_params:.2%})")
print(f"合并前后输出差异: {merge_diff:.6f}%")
print(f"\n关键: 合并后推理无额外开销，与原始线性层完全等价")

print(f"\n--- 不同rank的参数量对比 ---")
for r in [1, 2, 4, 8, 16, 32, 64]:
    lora_p = in_f * r + out_f * r
    print(f"  rank={r:<3} LoRA参数={lora_p:<8} 占原始{base_params}的{lora_p/base_params:.2%}")

### LoRA微调效果验证

#### 验证什么？

验证LoRA微调的实际效果：仅训练少量LoRA参数，观察模型性能变化，以及合并后推理是否零开销。

#### 验证要点

1. **参数量对比**：LoRA参数量 vs 原始模型参数量，通常<1%
2. **性能对比**：LoRA微调前后的loss/精度变化
3. **合并验证**：合并$W' = W + \frac{\alpha}{r}BA$后，输出与未合并是否一致
4. **推理速度**：合并后的推理速度应与原始模型完全相同

In [ ]:
class LoRAModel(nn.Module):
    """使用LoRA的简单模型"""
    def __init__(self, dim=64, n_classes=10, rank=8, alpha=16):
        super().__init__()
        self.layer1 = LoRALinear(dim, dim*2, rank=rank, alpha=alpha)
        self.layer2 = LoRALinear(dim*2, dim, rank=rank, alpha=alpha)
        self.layer3 = LoRALinear(dim, n_classes, rank=rank, alpha=alpha)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

    def lora_parameters(self):
        return [p for n, p in self.named_parameters() if 'lora_' in n]

    def freeze_base(self):
        for n, p in self.named_parameters():
            if 'lora_' not in n:
                p.requires_grad = False

dim, n_classes = 64, 10
x_data = torch.randn(256, dim)
y_data = torch.randint(0, n_classes, (256,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=32)

model_lora = LoRAModel(dim=dim, n_classes=n_classes, rank=8, alpha=16)
model_lora.freeze_base()

lora_params = sum(p.numel() for p in model_lora.lora_parameters())
total_params = sum(p.numel() for p in model_lora.parameters())

optimizer = torch.optim.Adam(model_lora.lora_parameters(), lr=1e-3)

losses = []
for epoch in range(30):
    for x, y in loader:
        out = model_lora(x)
        loss = F.cross_entropy(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

with torch.no_grad():
    acc_lora = (model_lora(x_data).argmax(1) == y_data).float().mean()

model_lora.layer1.merge_weights()
model_lora.layer2.merge_weights()
model_lora.layer3.merge_weights()

with torch.no_grad():
    acc_merged = (model_lora(x_data).argmax(1) == y_data).float().mean()

print(f"=== LoRA微调效果 ===")
print(f"总参数量: {total_params:,}")
print(f"LoRA可训练参数: {lora_params:,} ({lora_params/total_params:.2%})")
print(f"冻结参数: {total_params - lora_params:,}")
print(f"微调后准确率: {acc_lora:.4f}")
print(f"合并后准确率: {acc_merged:.4f}")
print(f"训练损失: {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"\nLoRA核心优势: 仅训练极少参数，合并后推理零开销")

### 低秩分解方法综合对比

#### 不同低秩分解方法对比

| 方法 | 原理 | 推理开销 | 是否需微调 | 适用场景 |
|------|------|---------|-----------|----------|
| 截断SVD | 保留前r个奇异值 | 两次小矩阵乘 | 可选 | 通用压缩 |
| Tucker分解 | 多模式分解 | 核张量+因子矩阵 | 可选 | 卷积核压缩 |
| LoRA | 低秩增量$\Delta W=BA$ | 合并后零开销 | 必须 | 微调+压缩 |

#### 选择指南

- **仅需压缩**：截断SVD，无需训练数据
- **需微调适配**：LoRA，训练效率最高
- **卷积模型**：Tucker分解，保留空间结构

In [ ]:
print(f"{'方法':<25} {'适用场景':<25} {'推理开销':<15} {'需要微调':<10}")
print("-" * 75)
methods = [
    ("SVD截断", "权重矩阵压缩", "减少(两步矩阵乘)", "可选"),
    ("Tucker分解", "高维张量(如Conv)", "减少", "可选"),
    ("LoRA重参数化", "微调/增量压缩", "零(合并后)", "必须"),
]
for name, scene, overhead, finetune in methods:
    print(f"{name:<25} {scene:<25} {overhead:<15} {finetune:<10}")

print(f"\n=== 产业级选择建议 ===")
print(f"1. 纯压缩(无训练): SVD截断，选择合适的rank保留能量")
print(f"2. 微调+压缩: LoRA，仅训练低秩参数，合并后零开销")
print(f"3. 卷积层压缩: Tucker分解，各维度独立选择秩")
print(f"4. 联合优化: 量化 + 低秩分解，先SVD压缩再量化，效果叠加")

m, n = 256, 512
W = torch.randn(m, n)
print(f"\n--- 压缩比对比 (矩阵 {m}x{n}) ---")
for r in [8, 16, 32, 64]:
    svd_params = m * r + r * n
    lora_params = m * r + r * n
    print(f"  rank={r}: 原始{m*n}参数 -> 压缩后{svd_params}参数 ({svd_params/(m*n):.2%}), 压缩比{m*n/svd_params:.1f}x")